# Polars + PySpark Cardinality Integration Validation

This notebook validates the **real runtime integrations** for:

- Polars `DataFrame`
- Polars `LazyFrame`
- PySpark `DataFrame`

It focuses on backend integration rather than the pandas/core manual validation notebook.

## What this validates

### Polars
- backend detection
- exact cardinality
- forced HLL promotion
- LazyFrame exact mode
- LazyFrame HLL mode
- null / NaN / signed-zero semantics
- whole-DataFrame profiling
- unsupported nested-column isolation
- all-unsupported row-count handling

### PySpark
- real `pyspark` package resolution
- Java diagnostics
- isolated Spark startup preflight
- backend detection
- exact cardinality
- distributed HLL promotion
- null / NaN / signed-zero semantics
- scalar dtype families
- whole-DataFrame profiling
- unsupported nested-column isolation
- all-unsupported row-count handling
- JSON lineage

> Recommended location: repository root, beside `pyproject.toml`.


## 0. Optional install

In [ ]:
# Uncomment if this kernel is not already using the current project.
# After installation, restart the kernel.
#
# %pip install -e ".[all-backends,test]"


## 1. Shared imports and configuration

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import importlib.util

import polars as pl

from data_quality import (
    AdaptiveCardinalityConfig,
    identify_backend,
    profile_cardinality,
    profile_cardinality_json,
)

EXACT_CONFIG = AdaptiveCardinalityConfig(
    exact_unique_threshold=50_000,
    hll_precision=14,
)

FORCE_HLL_CONFIG = AdaptiveCardinalityConfig(
    exact_unique_threshold=100,
    hll_precision=14,
)

def assert_hll_close(actual: int, expected: int, tolerance: float = 0.05):
    error = abs(actual - expected) / expected
    print("Expected:", expected)
    print("Estimated:", actual)
    print(f"Relative error: {error:.4%}")
    assert error < tolerance

print("Python:", sys.version.split()[0])
print("Polars:", pl.__version__)


# Polars real-runtime validation

## 2. Backend detection

In [ ]:
df = pl.DataFrame({"x": [1, 2, 3]})

backend = identify_backend(df)

print(backend)
print("Detected backend:", backend.name.value)

assert backend.name.value == "polars"

print("PASS: Polars backend detection")


## 3. Exact cardinality

In [ ]:
df = pl.DataFrame({
    "customer_id": [1, 2, 2, 3, None]
})

result = profile_cardinality(
    df,
    "customer_id",
    config=EXACT_CONFIG,
)

print("Count:", result.distinct_count)
print("Exact:", result.is_exact)
print("Rows:", result.total_row_count)
print("Non-null:", result.non_null_count)
print("Null:", result.null_count)

assert result.distinct_count == 3
assert result.is_exact is True
assert result.total_row_count == 5
assert result.non_null_count == 4
assert result.null_count == 1

print("PASS: Polars exact cardinality")


## 4. Forced HLL promotion

In [ ]:
expected = 100_000

df = pl.DataFrame({
    "customer_id": range(expected)
})

result = profile_cardinality(
    df,
    "customer_id",
    config=FORCE_HLL_CONFIG,
)

print("Estimate:", result.distinct_count)
print("Exact:", result.is_exact)
print("Promotion:", result.promotion)
print("Raw HLL:", result.raw_hll_estimate)

assert result.is_exact is False
assert result.promotion.occurred is True
assert result.raw_hll_estimate is not None

assert_hll_close(result.distinct_count, expected)

print("PASS: Polars HLL promotion")


## 5. LazyFrame exact mode

In [ ]:
lazy_df = pl.DataFrame({
    "customer_id": [1, 2, 2, 3, None]
}).lazy()

backend = identify_backend(lazy_df)

print("Detected backend:", backend.name.value)
assert backend.name.value == "polars"

result = profile_cardinality(
    lazy_df,
    "customer_id",
)

print("Count:", result.distinct_count)
print("Exact:", result.is_exact)

assert result.distinct_count == 3
assert result.is_exact is True

print("PASS: Polars LazyFrame exact")


## 6. LazyFrame HLL mode

In [ ]:
expected = 100_000

lazy_df = pl.DataFrame({
    "customer_id": range(expected)
}).lazy()

result = profile_cardinality(
    lazy_df,
    "customer_id",
    config=FORCE_HLL_CONFIG,
)

print("Estimate:", result.distinct_count)
print("Exact:", result.is_exact)
print("Promotion:", result.promotion)

assert result.is_exact is False
assert result.promotion.occurred is True

assert_hll_close(result.distinct_count, expected)

print("PASS: Polars LazyFrame HLL")


## 7. Null / NaN / signed-zero semantics

In [ ]:
df = pl.DataFrame({
    "x": [
        1.0,
        float("nan"),
        None,
        2.0,
        -0.0,
        0.0,
    ]
})

result = profile_cardinality(df, "x")

print("Count:", result.distinct_count)
print("Rows:", result.total_row_count)
print("Non-null:", result.non_null_count)
print("Null-like:", result.null_count)

assert result.distinct_count == 3
assert result.total_row_count == 6
assert result.null_count == 2

print("PASS: Polars null/NaN/signed-zero semantics")


## 8. Whole-DataFrame profiling

In [ ]:
df = pl.DataFrame({
    "id": [1, 2, 3, 3],
    "name": ["a", "b", "a", "c"],
    "score": [1.0, 2.0, 2.0, 3.0],
})

report = profile_cardinality(df)

print("Rows:", report.total_row_count)
print("Columns:", report.column_count)
print("Successful:", report.successful_column_count)
print("Unsupported:", report.unsupported_column_count)

for outcome in report.columns:
    print(
        outcome.column,
        "->",
        outcome.status,
        outcome.result.distinct_count if outcome.result is not None else None,
    )

assert report.total_row_count == 4
assert report.column_count == 3
assert report.successful_column_count == 3
assert report.unsupported_column_count == 0

print("PASS: Polars whole-DataFrame profiling")


## 9. Unsupported nested-column isolation

In [ ]:
df = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["a", "b", "a"],
    "nested": [
        [1, 2],
        [3],
        [4, 5],
    ],
})

report = profile_cardinality(df)

outcomes = {
    outcome.column: outcome
    for outcome in report.columns
}

print(report.to_json())

assert outcomes["id"].status == "ok"
assert outcomes["name"].status == "ok"
assert outcomes["nested"].status == "unsupported"

assert report.total_row_count == 3
assert report.successful_column_count == 2
assert report.unsupported_column_count == 1

print("PASS: Polars nested-column isolation")


## 10. All-unsupported row-count edge case

In [ ]:
df = pl.DataFrame({
    "a": [[1], [2], [3]],
    "b": [
        {"x": 1},
        {"x": 2},
        {"x": 3},
    ],
})

report = profile_cardinality(df)

print("Rows:", report.total_row_count)
print("Successful:", report.successful_column_count)
print("Unsupported:", report.unsupported_column_count)

assert report.total_row_count == 3
assert report.successful_column_count == 0
assert report.unsupported_column_count == 2

print("PASS: Polars all-unsupported row-count handling")


## 11. Polars JSON lineage

In [ ]:
df = pl.DataFrame({
    "customer_id": [1, 2, 2, 3],
})

payload = profile_cardinality_json(
    df,
    "customer_id",
)

parsed = json.loads(payload)

print(json.dumps(parsed, indent=2))

assert parsed["output"]["distinct_count"] == 3
assert parsed["lineage"]["backend"]["name"] == "polars"

print("PASS: Polars JSON lineage")


# PySpark real-runtime validation

PySpark is gated because Spark starts a JVM and can be slow or hang during environment setup.

Set `RUN_PYSPARK = True` below when ready.

The isolated startup test uses a configurable timeout. The default is **300 seconds**.


## 12. PySpark gate and timeout

In [ ]:
RUN_PYSPARK = False

SPARK_PREFLIGHT_TIMEOUT = 300

PYSPARK_AVAILABLE = False
SPARK_PREFLIGHT_OK = False
spark = None

print("RUN_PYSPARK:", RUN_PYSPARK)
print("SPARK_PREFLIGHT_TIMEOUT:", SPARK_PREFLIGHT_TIMEOUT)


## 13. PySpark + Java diagnostics

In [ ]:
if RUN_PYSPARK:
    spec = importlib.util.find_spec("pyspark")
    java_path = shutil.which("java")

    print("Python executable:", sys.executable)
    print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
    print("java executable:", java_path)
    print("pyspark origin:", spec.origin if spec else None)
    print(
        "pyspark package locations:",
        spec.submodule_search_locations if spec else None,
    )

    assert spec is not None, "PySpark is not installed."
    assert spec.submodule_search_locations is not None, (
        "Python found a plain module named pyspark, not the real package."
    )
    assert java_path is not None, "Java is not available on PATH."

    java_result = subprocess.run(
        ["java", "-version"],
        capture_output=True,
        text=True,
        timeout=10,
    )

    print("\njava -version:")
    print(java_result.stdout)
    print(java_result.stderr)

    PYSPARK_AVAILABLE = True
else:
    print("SKIP: PySpark diagnostics")


## 14. Isolated Spark startup preflight

In [ ]:
if RUN_PYSPARK and PYSPARK_AVAILABLE:
    preflight_code = r'''
import os

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

print("[1] process started", flush=True)

from pyspark.sql import SparkSession

print("[2] SparkSession imported", flush=True)

spark = (
    SparkSession.builder
    .master("local[1]")
    .appName("data-quality-preflight")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

print("[3] Spark session created", flush=True)

assert spark.range(1).count() == 1

print("[4] first Spark job completed", flush=True)
print("SPARK_PREFLIGHT_OK", flush=True)

spark.stop()
'''

    env = os.environ.copy()
    env["SPARK_LOCAL_IP"] = "127.0.0.1"

    print(
        "Starting isolated Spark preflight with timeout:",
        SPARK_PREFLIGHT_TIMEOUT,
        "seconds",
    )

    try:
        proc = subprocess.run(
            [sys.executable, "-c", preflight_code],
            capture_output=True,
            text=True,
            timeout=SPARK_PREFLIGHT_TIMEOUT,
            env=env,
        )

        print("Return code:", proc.returncode)

        if proc.stdout:
            print("\nstdout:")
            print(proc.stdout)

        if proc.stderr:
            print("\nstderr:")
            print(proc.stderr)

        SPARK_PREFLIGHT_OK = (
            proc.returncode == 0
            and "SPARK_PREFLIGHT_OK" in proc.stdout
        )

    except subprocess.TimeoutExpired as exc:
        SPARK_PREFLIGHT_OK = False

        print(
            "FAIL: Spark preflight exceeded",
            SPARK_PREFLIGHT_TIMEOUT,
            "seconds and was killed.",
        )

        if exc.stdout:
            print("\nPartial stdout:")
            print(exc.stdout)

        if exc.stderr:
            print("\nPartial stderr:")
            print(exc.stderr)

    print("\nSPARK_PREFLIGHT_OK =", SPARK_PREFLIGHT_OK)

else:
    print("SKIP: Spark startup preflight")


## 15. Start notebook Spark session

In [ ]:
if RUN_PYSPARK and SPARK_PREFLIGHT_OK:
    os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

    import pyspark
    from pyspark.sql import SparkSession
    from pyspark.sql import types as T

    print("PySpark:", pyspark.__version__)
    print("PySpark loaded from:", pyspark.__file__)

    spark = (
        SparkSession.builder
        .master("local[2]")
        .appName("data-quality-integration-validation")
        .config("spark.ui.enabled", "false")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")

    print("Spark version:", spark.version)
    print("Spark session started")
else:
    print("SKIP: Spark session start")


## 16. Spark runtime sanity check

In [ ]:
if spark is not None:
    count = spark.range(10).count()

    print("spark.range(10).count() =", count)

    assert count == 10

    print("PASS: Spark runtime")
else:
    print("SKIP: Spark runtime sanity check")


## 17. PySpark backend detection

In [ ]:
if spark is not None:
    df = spark.createDataFrame(
        [(1,), (2,), (3,)],
        ["x"],
    )

    backend = identify_backend(df)

    print(backend)
    print("Detected backend:", backend.name.value)

    assert backend.name.value == "pyspark"

    print("PASS: PySpark backend detection")
else:
    print("SKIP: PySpark backend detection")


## 18. PySpark exact cardinality

In [ ]:
if spark is not None:
    schema = T.StructType([
        T.StructField(
            "customer_id",
            T.LongType(),
            True,
        )
    ])

    df = spark.createDataFrame(
        [
            (1,),
            (2,),
            (2,),
            (3,),
            (None,),
        ],
        schema=schema,
    )

    result = profile_cardinality(
        df,
        "customer_id",
        config=EXACT_CONFIG,
    )

    print("Count:", result.distinct_count)
    print("Exact:", result.is_exact)
    print("Rows:", result.total_row_count)
    print("Non-null:", result.non_null_count)
    print("Null:", result.null_count)

    assert result.distinct_count == 3
    assert result.is_exact is True
    assert result.total_row_count == 5
    assert result.non_null_count == 4
    assert result.null_count == 1

    print("PASS: PySpark exact cardinality")
else:
    print("SKIP: PySpark exact cardinality")


## 19. PySpark distributed HLL promotion

In [ ]:
if spark is not None:
    expected = 100_000

    df = (
        spark.range(expected)
        .withColumnRenamed(
            "id",
            "customer_id",
        )
    )

    result = profile_cardinality(
        df,
        "customer_id",
        config=FORCE_HLL_CONFIG,
    )

    print("Estimate:", result.distinct_count)
    print("Exact:", result.is_exact)
    print("Promotion:", result.promotion)
    print("Raw HLL:", result.raw_hll_estimate)
    print("Lineage:", result.lineage)

    assert result.is_exact is False
    assert result.promotion.occurred is True
    assert result.raw_hll_estimate is not None

    assert_hll_close(
        result.distinct_count,
        expected,
    )

    print("PASS: PySpark distributed HLL promotion")
else:
    print("SKIP: PySpark distributed HLL promotion")


## 20. PySpark null / NaN / signed-zero semantics

In [ ]:
if spark is not None:
    schema = T.StructType([
        T.StructField(
            "x",
            T.DoubleType(),
            True,
        )
    ])

    df = spark.createDataFrame(
        [
            (1.0,),
            (float("nan"),),
            (None,),
            (2.0,),
            (-0.0,),
            (0.0,),
        ],
        schema=schema,
    )

    result = profile_cardinality(
        df,
        "x",
    )

    print("Count:", result.distinct_count)
    print("Rows:", result.total_row_count)
    print("Null-like:", result.null_count)

    assert result.distinct_count == 3
    assert result.total_row_count == 6
    assert result.null_count == 2

    print("PASS: PySpark null/NaN/signed-zero")
else:
    print("SKIP: PySpark null/NaN/signed-zero")


## 21. PySpark scalar dtype families

In [ ]:
if spark is not None:
    schema = T.StructType([
        T.StructField("integer_col", T.LongType(), True),
        T.StructField("float_col", T.DoubleType(), True),
        T.StructField("bool_col", T.BooleanType(), True),
        T.StructField("string_col", T.StringType(), True),
        T.StructField("binary_col", T.BinaryType(), True),
    ])

    df = spark.createDataFrame(
        [
            (1, 1.0, True, "a", b"a"),
            (2, 2.0, False, "b", b"b"),
            (3, -0.0, True, "a", b"a"),
            (3, 0.0, False, "c", b"c"),
        ],
        schema=schema,
    )

    expected = {
        "integer_col": 3,
        "float_col": 3,
        "bool_col": 2,
        "string_col": 3,
        "binary_col": 3,
    }

    for column, expected_count in expected.items():
        result = profile_cardinality(df, column)
        print(
            column,
            "->",
            result.distinct_count,
            "| family:",
            result.lineage.get("dtype_family"),
        )
        assert result.distinct_count == expected_count

    print("PASS: PySpark scalar dtype families")
else:
    print("SKIP: PySpark scalar dtype families")


## 22. PySpark whole-DataFrame profiling

In [ ]:
if spark is not None:
    report = profile_cardinality(df)

    print("Rows:", report.total_row_count)
    print("Columns:", report.column_count)
    print("Successful:", report.successful_column_count)
    print("Unsupported:", report.unsupported_column_count)

    assert report.total_row_count == 4
    assert report.column_count == 5
    assert report.successful_column_count == 5
    assert report.unsupported_column_count == 0

    print("PASS: PySpark whole-DataFrame profiling")
else:
    print("SKIP: PySpark whole-DataFrame profiling")


## 23. PySpark unsupported nested-column isolation

In [ ]:
if spark is not None:
    schema = T.StructType([
        T.StructField(
            "id",
            T.LongType(),
            False,
        ),
        T.StructField(
            "nested",
            T.ArrayType(T.LongType()),
            True,
        ),
    ])

    df = spark.createDataFrame(
        [
            (1, [1, 2]),
            (2, [3]),
            (3, [4, 5]),
        ],
        schema=schema,
    )

    report = profile_cardinality(df)

    outcomes = {
        outcome.column: outcome
        for outcome in report.columns
    }

    print(report.to_json())

    assert outcomes["id"].status == "ok"
    assert outcomes["nested"].status == "unsupported"

    assert report.total_row_count == 3
    assert report.successful_column_count == 1
    assert report.unsupported_column_count == 1

    print("PASS: PySpark nested-column isolation")
else:
    print("SKIP: PySpark nested-column isolation")


## 24. PySpark all-unsupported row-count edge case

In [ ]:
if spark is not None:
    schema = T.StructType([
        T.StructField(
            "array_col",
            T.ArrayType(T.IntegerType()),
            True,
        ),
        T.StructField(
            "map_col",
            T.MapType(
                T.StringType(),
                T.IntegerType(),
            ),
            True,
        ),
    ])

    df = spark.createDataFrame(
        [
            ([1], {"a": 1}),
            ([2], {"b": 2}),
            ([3], {"c": 3}),
        ],
        schema=schema,
    )

    report = profile_cardinality(df)

    print("Rows:", report.total_row_count)
    print("Successful:", report.successful_column_count)
    print("Unsupported:", report.unsupported_column_count)

    assert report.total_row_count == 3
    assert report.successful_column_count == 0
    assert report.unsupported_column_count == 2

    print("PASS: PySpark all-unsupported row-count handling")
else:
    print("SKIP: PySpark all-unsupported row-count handling")


## 25. PySpark JSON lineage

In [ ]:
if spark is not None:
    df = spark.createDataFrame(
        [
            (1,),
            (2,),
            (2,),
            (3,),
        ],
        ["customer_id"],
    )

    payload = profile_cardinality_json(
        df,
        "customer_id",
    )

    parsed = json.loads(payload)

    print(json.dumps(parsed, indent=2))

    assert parsed["output"]["distinct_count"] == 3
    assert parsed["lineage"]["backend"]["name"] == "pyspark"

    print("PASS: PySpark JSON lineage")
else:
    print("SKIP: PySpark JSON lineage")


# Cross-backend consistency

## 26. Polars vs PySpark exact-count consistency

In [ ]:
values = [1, 2, 2, 3, 4, 4, 5]

polars_df = pl.DataFrame({
    "id": values,
})

polars_result = profile_cardinality(
    polars_df,
    "id",
)

print("Polars:", polars_result.distinct_count)

assert polars_result.distinct_count == 5

if spark is not None:
    spark_df = spark.createDataFrame(
        [(value,) for value in values],
        ["id"],
    )

    spark_result = profile_cardinality(
        spark_df,
        "id",
    )

    print("PySpark:", spark_result.distinct_count)

    assert spark_result.distinct_count == 5

    print("PASS: Polars / PySpark exact consistency")
else:
    print("PySpark skipped")
    print("PASS: Polars exact consistency")


# Automated adapter-specific pytest checks

## 27. Polars tests

In [ ]:
# Uncomment to run:
# !python -m pytest tests/cardinality -q -k "polars"


## 28. PySpark tests

In [ ]:
# Uncomment to run:
# !python -m pytest tests/cardinality -q -k "pyspark or spark"


## 29. Full cardinality regression

In [ ]:
# Uncomment to run:
# !python -m pytest tests/cardinality -q


# Cleanup

In [ ]:
if spark is not None:
    spark.stop()
    spark = None
    print("Spark stopped")
else:
    print("No Spark session to stop")


# Final status

In [ ]:
print("===========================================")
print("POLARS + PYSPARK INTEGRATION RUN COMPLETED")
print("===========================================")
print("Polars: executed")
print(
    "PySpark:",
    "executed"
    if RUN_PYSPARK and SPARK_PREFLIGHT_OK
    else "skipped",
)

if not RUN_PYSPARK:
    print(
        "\nTo run PySpark:"
        "\n1. Set RUN_PYSPARK = True in section 12."
        "\n2. Run sections 13 onward."
        "\n3. Increase SPARK_PREFLIGHT_TIMEOUT if startup is slow."
    )
